# Loss functions and gradient clipping for decision models, on a small network

A learning notebook. Signpost-D answers three kinds of question with a probability for every option:
**yes/no** statements, **choices** between named options, and **scores** on an ordered scale. It is trained against a
*soft* gold: the averaged judgement of several judges, which is decisive when the case is clear and hedged when it is not.

Two choices shape how such a model learns:

- **the loss**: how a prediction is compared with the gold distribution;
- **the gradient clipper**: what happens when one batch produces a much larger gradient than usual.

They work together: the loss decides *which way* and *how hard* each example pulls; the clipper decides how much of that pull
reaches the weights. This notebook builds a tiny model of the same shape on synthetic data, so every loss and clipper can be
tried in seconds on a CPU and compared in plots.

**Contents:** 1 · setup · 2 · the data · 3 · the model · 4 · the losses · 5 · what each loss wants (no network) ·
6 · the clippers · 7 · metrics and the training loop · 8 · experiments with plots · 9 · what to take away

## 1. Setup

Plain PyTorch, matplotlib for the plots and pandas for the tables. Everything runs on a CPU. `KINDS` are the three question
types, `N_OPT` how many options each has, `D` the size of the input features that stand in for a case's state.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(0)
KINDS = ["noul", "choice", "score"]          # yes/no, choice, ordered score
N_OPT = {"noul": 2, "choice": 4, "score": 5}
D = 16                                       # input features per case

## 2. The data

### A hidden "teacher"

The gold comes from a fixed, hidden rule (the teacher): random linear maps from the features to each question's options, and
for scores a single number that is placed on a 5-level scale. The network never sees the teacher; it only sees the features
and the gold distributions, as Signpost-D only sees states and judges' answers.

In [ ]:
teacher_rng = torch.Generator().manual_seed(1234)
TEACHER = {
    "noul": torch.randn(D, 2, generator=teacher_rng) / math.sqrt(D),
    "choice": torch.randn(D, 4, generator=teacher_rng) / math.sqrt(D),
    "score": torch.randn(D, generator=teacher_rng) / math.sqrt(D),
}
LEVEL_CENTRES = torch.linspace(-1.5, 1.5, 5)   # where each of the 5 score levels sits on the teacher's number line

### Clear cases, ambiguous cases and traps

`make_split` builds a set of cases, each with one question of every kind:

- **Clear cases (70%)**: the teacher's logits are multiplied by 4, so the gold is decisive (often 0.95 or more on one
  option). For scores, the gold is a narrow bump around the true level.
- **Ambiguous cases (30%)**: the logits are multiplied by 0.4, so the gold hedges; score gold is a wide bump spread over
  neighbouring levels.
- **Traps (5% of training cases)**: the gold is reversed (yes ↔ no, choices and levels in reverse order), so it contradicts
  the evidence, and their inputs are 3× larger than usual. These are the rare examples that produce **gradient spikes**,
  like a mislabelled record or an unusually long state in real data.

The dev set has no traps: it measures what the model learned from the honest cases.

In [ ]:
def make_split(n, seed, clear_share=0.7, trap_share=0.05):
    rng = torch.Generator().manual_seed(seed)
    x = torch.randn(n, D, generator=rng)
    clear = torch.rand(n, generator=rng) < clear_share
    trap = torch.rand(n, generator=rng) < trap_share
    sharp = torch.where(clear, torch.tensor(4.0), torch.tensor(0.4))[:, None]
    split = {"clear": clear, "trap": trap}
    split["noul"] = torch.softmax(x @ TEACHER["noul"] * sharp, -1)
    split["choice"] = torch.softmax(x @ TEACHER["choice"] * sharp, -1)
    s = (x @ TEACHER["score"])[:, None]
    width = torch.where(clear, torch.tensor(0.3), torch.tensor(0.9))[:, None]
    split["score"] = torch.softmax(-(s - LEVEL_CENTRES) ** 2 / (2 * width ** 2), -1)
    for k in KINDS:                                   # traps: the gold contradicts the evidence
        split[k][trap] = split[k][trap].flip(-1)
    x[trap] *= 3.0                                    # ... and their inputs are unusually large
    split["x"] = x
    return split

### The two sets

4,000 training cases and 1,000 dev cases. The printout checks the mix: about 70% clear, about 5% traps in training, and the
share of yes/no gold that is decisive (0.85 or more on one side).

In [ ]:
train_set = make_split(4000, seed=1)
dev_set = make_split(1000, seed=2, trap_share=0.0)
N_TRAIN = len(train_set["x"])

yes = train_set["noul"][:, 0]
print(f"train {N_TRAIN} cases, dev {len(dev_set['x'])} cases")
print(f"clear {train_set['clear'].float().mean():.2f} | traps {train_set['trap'].float().mean():.3f} | "
      f"decisive yes/no gold {((yes >= 0.85) | (yes <= 0.15)).float().mean():.2f}")

### What the gold looks like

Left: how the yes/no gold is spread, with a peak at each end (clear cases) and a hump in the middle (ambiguous ones).
Right: the gold for one clear and one ambiguous score question.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(yes.numpy(), bins=40)
ax[0].set(title="yes/no gold: P(yes)", xlabel="P(yes)", ylabel="cases")
i_clear = int(torch.nonzero(train_set["clear"] & ~train_set["trap"])[0])
i_amb = int(torch.nonzero(~train_set["clear"] & ~train_set["trap"])[0])
w = 0.4
ax[1].bar(torch.arange(5) - w / 2, train_set["score"][i_clear], w, label="clear case")
ax[1].bar(torch.arange(5) + w / 2, train_set["score"][i_amb], w, label="ambiguous case")
ax[1].set(title="score gold over 5 levels", xlabel="level", ylabel="probability")
ax[1].legend()
plt.tight_layout()

## 3. The model

A shared trunk (two small layers) reads the features, and one output layer per question kind gives that kind's logits, the
way Signpost-D reads the state once and answers each question from the label tokens. Softmax over a head's logits gives the
predicted distribution.

In [ ]:
class Decider(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(D, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU())
        self.heads = nn.ModuleDict({k: nn.Linear(hidden, n) for k, n in N_OPT.items()})

    def forward(self, x):
        h = self.trunk(x)
        return {k: head(h) for k, head in self.heads.items()}

## 4. The losses

Every loss has the same signature, `loss(logits, gold, kind) -> one number` (the mean over the batch), so any of them can be
dropped into the training loop. `kind` lets a loss treat score questions differently.

The small helper below returns the log-probabilities and probabilities of a batch of logits; most losses need one or both.

In [ ]:
def logp_and_p(logits):
    logp = F.log_softmax(logits, -1)
    return logp, logp.exp()

### Soft cross-entropy (Signpost-D's base loss)

$-\sum_i g_i \log p_i$: the gold-weighted surprise of the prediction. Its minimum is exactly $p = g$, so the model is pushed
to copy the whole gold distribution, decisive or hedged. Minimising it is the same as minimising KL(gold ‖ prediction), since
the two differ by the gold's entropy, a constant.

In [ ]:
def soft_ce(logits, gold, kind):
    logp, _ = logp_and_p(logits)
    return -(gold * logp).sum(-1).mean()

### Soft cross-entropy + expected-level term (what Signpost-D uses)

For score questions, Signpost-D adds 0.5 × |expected level of the prediction − expected level of the gold|, divided by
(levels − 1). It rewards getting the *average* level right, which plain cross-entropy ignores (to cross-entropy, level 0
instead of 4 is no worse than 3 instead of 4). The weakness, shown in section 5: very different distributions can share
the same average.

In [ ]:
def expected_level_gap(logits, gold):
    _, p = logp_and_p(logits)
    levels = torch.arange(logits.shape[-1], dtype=logits.dtype)
    gap = ((p * levels).sum(-1) - (gold * levels).sum(-1)).abs()
    return (gap / (logits.shape[-1] - 1)).mean()


def ce_expected_level(logits, gold, kind):
    loss = soft_ce(logits, gold, kind)
    return loss + 0.5 * expected_level_gap(logits, gold) if kind == "score" else loss

### Soft cross-entropy + earth mover's distance (for ordered scales)

The squared earth mover's distance compares the *cumulative* distributions: $\sum_k (P_{\le k} - G_{\le k})^2$. Moving
probability one level away costs a little; moving it four levels away costs much more, so it respects the order of the
scale and cannot be fooled by mass split between both ends.

In [ ]:
def emd2(logits, gold):
    _, p = logp_and_p(logits)
    return ((p.cumsum(-1) - gold.cumsum(-1)) ** 2).sum(-1).mean()


def ce_emd(logits, gold, kind):
    loss = soft_ce(logits, gold, kind)
    return loss + emd2(logits, gold) if kind == "score" else loss

### Soft cross-entropy mixed with hard-label cross-entropy

$(1-\lambda)\,\text{soft CE} + \lambda\,\text{CE}(\text{gold's top option})$. Its minimum is the gold pulled a little toward
its top answer, $(1-\lambda)\,g + \lambda\,\text{one-hot}$: more decisive and more often right on the top answer, at a small
cost in KL on genuinely hedged cases. With λ = 0.15 a 0.6 / 0.4 gold becomes about 0.66 / 0.34.

In [ ]:
def ce_hard(logits, gold, kind, lam=0.15):
    hard = F.cross_entropy(logits, gold.argmax(-1))
    return (1 - lam) * soft_ce(logits, gold, kind) + lam * hard

### Soft cross-entropy + a margin (hinge) on the top answer

Adds `weight × max(0, margin − (top logit − best other logit))`: a penalty until the gold's top option leads every other
option by `margin` in logit. It aims straight at top-1 accuracy, the metric Typed Decisions ranks on. On its own it would
ignore the rest of the distribution, so here it is a small add-on to soft cross-entropy.

In [ ]:
def ce_margin(logits, gold, kind, margin=1.0, weight=0.2):
    top = gold.argmax(-1)
    top_logit = logits.gather(-1, top[:, None]).squeeze(-1)
    rest = logits.masked_fill(F.one_hot(top, logits.shape[-1]).bool(), float("-inf")).max(-1).values
    return soft_ce(logits, gold, kind) + weight * F.relu(margin - (top_logit - rest)).mean()

### Brier loss

$\sum_i (p_i - g_i)^2$: squared error between the two distributions. It is bounded (at most 2), so a badly wrong example
can never produce a huge loss or gradient, and it is gentle on options whose gold is tiny. The price: near a confident
mistake its gradient is *small*, so it corrects such mistakes slowly.

In [ ]:
def brier(logits, gold, kind):
    _, p = logp_and_p(logits)
    return ((p - gold) ** 2).sum(-1).mean()

### Reverse KL and Jensen–Shannon

- **Reverse KL**, KL(prediction ‖ gold) $=\sum_i p_i(\log p_i - \log g_i)$: it punishes the prediction for putting mass where
  the gold has little, so a model that cannot fit the gold exactly prefers to sit on its main answer ("mode-seeking").
- **Jensen–Shannon**: the average KL of each to their midpoint. Symmetric and bounded, but its gradients are small, so it
  learns slowly.

In [ ]:
def reverse_kl(logits, gold, kind):
    logp, p = logp_and_p(logits)
    return (p * (logp - gold.clamp_min(1e-6).log())).sum(-1).mean()


def jensen_shannon(logits, gold, kind):
    logp, p = logp_and_p(logits)
    m = (p + gold) / 2
    logm = m.clamp_min(1e-9).log()
    kl_pm = (p * (logp - logm)).sum(-1)
    kl_gm = (gold * (gold.clamp_min(1e-9).log() - logm)).sum(-1)
    return (0.5 * kl_pm + 0.5 * kl_gm).mean()

### Focal soft cross-entropy

$-\sum_i g_i (1-p_i)^\gamma \log p_i$: each option's term is scaled down when the model already gives it high probability, so
easy, already-learned questions count for little and hard ones dominate. Useful when a few families are hard; noisier,
because traps are "hard" too.

In [ ]:
def focal_ce(logits, gold, kind, gamma=2.0):
    logp, p = logp_and_p(logits)
    return -(gold * (1 - p) ** gamma * logp).sum(-1).mean()

### All losses in one place

A name → function table, used by every experiment below. (Per-type weights, the last idea on the list, are a parameter of the
training loop rather than a loss: `type_weights={"score": 2.0}` doubles the score questions' share.)

In [ ]:
LOSSES = {
    "soft_ce": soft_ce,
    "ce_expected_level": ce_expected_level,
    "ce_emd": ce_emd,
    "ce_hard": ce_hard,
    "ce_margin": ce_margin,
    "brier": brier,
    "reverse_kl": reverse_kl,
    "jensen_shannon": jensen_shannon,
    "focal_ce": focal_ce,
}

## 5. What each loss wants, without a network

Before training anything, two small experiments show each loss's character directly.

### Loss and gradient for one yes/no question

For a single yes/no question we sweep the predicted P(yes) from 0.01 to 0.99, and for each value compute the loss and its
gradient with respect to the "yes" logit. Gold is either decisive (0.9) or hedged (0.6). `curve` does one sweep; each
position is an independent one-question batch, so summing before `backward()` gives every position its own gradient.

In [ ]:
P_GRID = torch.linspace(0.01, 0.99, 197)


def curve(loss_fn, gold_yes):
    z = torch.stack([torch.logit(P_GRID), torch.zeros_like(P_GRID)], -1).requires_grad_(True)
    gold = torch.tensor([gold_yes, 1 - gold_yes])
    per_item = torch.stack([loss_fn(z[i:i + 1], gold[None], "noul") for i in range(len(P_GRID))])
    per_item.sum().backward()
    return per_item.detach(), z.grad[:, 0].detach()

### Plot: the shapes of the losses

Top row: each loss minus its own minimum, so the curves can share an axis. The dashed line marks the gold. Bottom row: the
gradient on the yes logit (negative means "raise P(yes)"). Things to look for:

- soft CE, Brier and JS have their minimum at the gold itself; **ce_hard** and **ce_margin** have theirs to the right of it,
  more decisive than the gold;
- near a confident mistake (P(yes) close to 0 when the gold is 0.9), cross-entropy's gradient stays strong, while Brier's and
  JS's fade away;
- focal CE is flat around the gold: it barely cares about questions it already gets roughly right.

(The score-only variants are identical to soft CE for yes/no, so they are left out.)

In [ ]:
SHOW = ["soft_ce", "ce_hard", "ce_margin", "brier", "reverse_kl", "jensen_shannon", "focal_ce"]
fig, ax = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for col, gold_yes in enumerate([0.9, 0.6]):
    for name in SHOW:
        loss, grad = curve(LOSSES[name], gold_yes)
        ax[0, col].plot(P_GRID, loss - loss.min(), label=name)
        ax[1, col].plot(P_GRID, grad, label=name)
    for row in range(2):
        ax[row, col].axvline(gold_yes, ls="--", c="grey")
    ax[0, col].set(title=f"gold P(yes) = {gold_yes}", ylabel="loss − its minimum", ylim=(0, 2))
    ax[1, col].set(xlabel="predicted P(yes)", ylabel="d loss / d yes-logit")
ax[0, 0].legend(fontsize=8)
plt.tight_layout()

### Where each loss settles on a hedged choice

Now a single choice question with hedged gold (0.50 / 0.30 / 0.15 / 0.05), and four free logits instead of a network.
`fit_free_logits` runs Adam on those logits under one loss and returns the distribution it settles on. With unlimited
freedom, any loss whose minimum is the gold will reproduce it; the ones that don't show their bias.

In [ ]:
def fit_free_logits(loss_fn, gold, kind, steps=2000, lr=0.05):
    z = torch.zeros(1, len(gold), requires_grad=True)
    opt = torch.optim.Adam([z], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(z, gold[None], kind).backward()
        opt.step()
    return z.detach().softmax(-1)[0]


HEDGED = torch.tensor([0.50, 0.30, 0.15, 0.05])
settled = {name: fit_free_logits(LOSSES[name], HEDGED, "choice") for name in SHOW}

### Plot: the gold against what each loss settles on

Bars per option: the gold first, then each loss. Expect soft CE, Brier, reverse KL and JS on the gold; ce_hard a little
sharper on option A; ce_margin sharper still (it keeps pushing until A leads B by the margin); focal CE flatter, because it
stops caring about options that are already likely.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
series = {"gold": HEDGED, **settled}
width = 0.8 / len(series)
for j, (name, dist) in enumerate(series.items()):
    ax.bar(torch.arange(4) + (j - len(series) / 2) * width, dist, width, label=name)
ax.set(xticks=range(4), xticklabels=["A", "B", "C", "D"], ylabel="probability",
       title="where each loss settles on a hedged choice (free logits)")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()

### The expected-level blind spot

Three predictions for a score question whose gold is concentrated on the middle level (level 2): one split between the two
ends (levels 0 and 4), one a level too low, and one equal to the gold. `as_logits` turns a probability vector into logits so
the loss functions can take it. The table shows that the **expected-level term rates the split prediction perfect** (both
average 2), while the earth mover's distance rates it the worst of the three.

In [ ]:
def as_logits(q):
    return q.clamp_min(1e-6).log()[None]


GOLD_MID = torch.tensor([0.0, 0.1, 0.8, 0.1, 0.0])
PREDS = {"split between the ends": torch.tensor([0.5, 0.0, 0.0, 0.0, 0.5]),
         "one level too low": torch.tensor([0.0, 0.8, 0.2, 0.0, 0.0]),
         "same as the gold": GOLD_MID}
pd.DataFrame({name: {"expected-level gap": expected_level_gap(as_logits(q), GOLD_MID[None]).item(),
                     "earth mover's distance²": emd2(as_logits(q), GOLD_MID[None]).item(),
                     "soft cross-entropy": soft_ce(as_logits(q), GOLD_MID[None], "score").item()}
              for name, q in PREDS.items()}).T.round(3)

## 6. The gradient clippers

After `backward()` and before the optimizer step, a clipper looks at the gradient and may shrink it. Each clipper below is a
small class whose call returns `(norm before, norm after, skipped)`. Two helpers first: `grad_norm` is the size of the whole
gradient (all parameters together), and `scale_grads` multiplies every gradient by one factor, which shortens the step but
keeps its direction.

In [ ]:
def grad_norm(params):
    return torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in params if p.grad is not None)).item()


def scale_grads(params, factor):
    for p in params:
        if p.grad is not None:
            p.grad.mul_(factor)

### No clipping, and a fixed norm limit (Signpost-D's setting)

- `NoClip` passes everything through: spikes reach the optimizer unchanged.
- `FixedNorm(max_norm)` is `torch.nn.utils.clip_grad_norm_`: if the gradient's norm is above `max_norm`, scale it down to
  exactly `max_norm`. If the *typical* norm is above the limit (Signpost-D's is about 3–5 against a limit of 1), almost
  every step is clipped, and the limit, not the gradient, sets the step length.

In [ ]:
class NoClip:
    def __call__(self, params):
        n = grad_norm(params)
        return n, n, False


class FixedNorm:
    def __init__(self, max_norm=1.0):
        self.max_norm = max_norm

    def __call__(self, params):
        n = grad_norm(params)
        if n > self.max_norm:
            scale_grads(params, self.max_norm / (n + 1e-12))
        return n, min(n, self.max_norm), False

### Clipping by value

`ValueClip(v)` caps every single gradient element at ±v. Unlike norm clipping, it changes the gradient's *direction*: large
components are cut, small ones are untouched. Simple, but it treats a whole spike as many small independent overshoots.

In [ ]:
class ValueClip:
    def __init__(self, value=0.05):
        self.value = value

    def __call__(self, params):
        n = grad_norm(params)
        for p in params:
            if p.grad is not None:
                p.grad.clamp_(-self.value, self.value)
        return n, grad_norm(params), False

### AutoClip: a limit that follows the run

`AutoClip(percentile)` keeps every norm seen so far and clips at a chosen percentile of that history (the 10th in the original
paper). There is no limit to choose by hand: it adapts as gradients shrink over training.

In [ ]:
class AutoClip:
    def __init__(self, percentile=10):
        self.percentile, self.history = percentile, []

    def __call__(self, params):
        n = grad_norm(params)
        self.history.append(n)
        limit = float(torch.tensor(self.history).quantile(self.percentile / 100))
        if n > limit:
            scale_grads(params, limit / (n + 1e-12))
        return n, min(n, limit), False

### ZClip: clip only the statistical outliers

`ZClip(z_max)` keeps a running mean and variance of the norm (exponential moving averages). For each step it computes the
z-score, how many standard deviations the norm is above the mean; only a step with z above `z_max` is pulled back, to
mean + z_max × std. Normal steps pass untouched, so the gradient keeps its natural size; only spikes are tamed. The first
`warmup` steps just collect norms to start the statistics, and the statistics are updated with the *clipped* norm, so one
spike does not inflate them.

In [ ]:
class ZClip:
    def __init__(self, z_max=2.5, alpha=0.97, warmup=25):
        self.z_max, self.alpha, self.warmup = z_max, alpha, warmup
        self.seen, self.mean, self.var = [], None, None

    def __call__(self, params):
        n = grad_norm(params)
        if self.mean is None:                          # warm-up: only collect norms
            self.seen.append(n)
            if len(self.seen) == self.warmup:
                t = torch.tensor(self.seen)
                self.mean, self.var = t.mean().item(), t.var().item()
            return n, n, False
        limit = self.mean + self.z_max * math.sqrt(self.var)
        post = n
        if n > limit:
            scale_grads(params, limit / (n + 1e-12))
            post = limit
        self.mean = self.alpha * self.mean + (1 - self.alpha) * post
        self.var = self.alpha * self.var + (1 - self.alpha) * (post - self.mean) ** 2
        return n, post, False

### Skipping spikes entirely

`SkipSpike(k)` uses the same running statistics, but an outlier (more than k standard deviations above the mean) is not
shrunk: the whole update is **skipped**, and the statistics are left as they were. Some large pretraining runs do this, on
the view that a spike batch carries mostly noise.

In [ ]:
class SkipSpike:
    def __init__(self, k=4.0, alpha=0.97, warmup=25):
        self.k, self.alpha, self.warmup = k, alpha, warmup
        self.seen, self.mean, self.var = [], None, None

    def __call__(self, params):
        n = grad_norm(params)
        if self.mean is None:
            self.seen.append(n)
            if len(self.seen) == self.warmup:
                t = torch.tensor(self.seen)
                self.mean, self.var = t.mean().item(), t.var().item()
            return n, n, False
        if n > self.mean + self.k * math.sqrt(self.var):
            return n, 0.0, True                         # skip this update
        self.mean = self.alpha * self.mean + (1 - self.alpha) * n
        self.var = self.alpha * self.var + (1 - self.alpha) * (n - self.mean) ** 2
        return n, n, False

### Adaptive gradient clipping (AGC)

`AGC(ratio)` clips each parameter tensor on its own, relative to that tensor's weights: a gradient may be at most
`ratio × ‖weights‖`. A big layer may take a big gradient; a small one may not. (The original method works per output unit;
per tensor is the same idea, simpler. It was designed for plain SGD; with Adam, which already rescales each weight, its
effect is smaller.)

In [ ]:
class AGC:
    def __init__(self, ratio=0.05, eps=1e-3):
        self.ratio, self.eps = ratio, eps

    def __call__(self, params):
        pre = grad_norm(params)
        for p in params:
            if p.grad is None:
                continue
            limit = self.ratio * max(p.detach().norm().item(), self.eps)
            g = p.grad.norm().item()
            if g > limit:
                p.grad.mul_(limit / (g + 1e-12))
        return pre, grad_norm(params), False

### All clippers in one place

Each entry *makes* a fresh clipper, because ZClip, AutoClip and SkipSpike keep state that must not carry over between runs.

In [ ]:
CLIPPERS = {
    "none": NoClip,
    "fixed_norm_1": lambda: FixedNorm(1.0),
    "value_0.05": lambda: ValueClip(0.05),
    "autoclip_p10": lambda: AutoClip(10),
    "zclip_2.5": lambda: ZClip(2.5),
    "skip_spikes_4sd": lambda: SkipSpike(4.0),
    "agc_0.05": lambda: AGC(0.05),
}

## 7. Metrics and the training loop

### Calibration error

`ece` is the expected calibration error: put predictions in 10 bins by their confidence (the top probability), and in each
bin compare the average confidence with how often the top answer is right. A model that says 0.8 and is right 80% of the
time scores 0.

In [ ]:
def ece(conf, correct, bins=10):
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        inside = (conf > lo) & (conf <= hi)
        if inside.any():
            total += inside.float().mean() * (conf[inside].mean() - correct[inside].float().mean()).abs()
    return float(total)

### Scoring a model on dev

`evaluate` returns one row of numbers, the ones our two benchmarks care about:

- `acc`, `kl`: top-answer accuracy and KL from the gold, averaged over the three kinds (Typed Decisions' main measures);
  also per kind;
- `acc_clear`, `kl_ambiguous`: accuracy on the clear cases and KL on the ambiguous ones, which separates "right when it is
  clear" from "honest when it is not";
- `score_level_err`: how far the predicted average level is from the gold's, for score questions;
- `ece`: calibration, over all questions;
- `yesno_decisive`, `yesno_decisive_right`: the share of yes/no answers outside 0.2–0.8, and the share that are decisive *and*
  on the gold's side (a JevBench-style count, where a hedge earns nothing).

In [ ]:
@torch.no_grad()
def evaluate(model, split):
    out = model(split["x"])
    clear = split["clear"]
    row, confs, oks, accs_clear, kls_amb = {}, [], [], [], []
    for k in KINDS:
        p, g = out[k].softmax(-1), split[k]
        ok = p.argmax(-1) == g.argmax(-1)
        kl = (g * (g.clamp_min(1e-9).log() - p.clamp_min(1e-9).log())).sum(-1)
        row[f"{k}_acc"], row[f"{k}_kl"] = ok.float().mean().item(), kl.mean().item()
        accs_clear.append(ok[clear].float().mean().item())
        kls_amb.append(kl[~clear].mean().item())
        confs.append(p.max(-1).values)
        oks.append(ok)
    row["acc"] = sum(row[f"{k}_acc"] for k in KINDS) / 3
    row["kl"] = sum(row[f"{k}_kl"] for k in KINDS) / 3
    row["acc_clear"], row["kl_ambiguous"] = sum(accs_clear) / 3, sum(kls_amb) / 3
    levels = torch.arange(5.0)
    p_score = out["score"].softmax(-1)
    row["score_level_err"] = ((p_score * levels).sum(-1) - (split["score"] * levels).sum(-1)).abs().mean().item()
    row["ece"] = ece(torch.cat(confs), torch.cat(oks))
    p_yes, g_yes = out["noul"].softmax(-1)[:, 0], split["noul"][:, 0]
    decisive = (p_yes < 0.2) | (p_yes > 0.8)
    row["yesno_decisive"] = decisive.float().mean().item()
    row["yesno_decisive_right"] = (decisive & ((p_yes > 0.5) == (g_yes > 0.5))).float().mean().item()
    return row

### The training loop

`train` trains one model with one loss and one clipper, and records what happens at every step:

- **The same start for every run:** the same seed gives the same initial weights and the same batches, so differences come
  only from the loss and the clipper.
- **AdamW** with a learning rate that warms up over the first 15% of steps and then falls linearly to zero, as in
  Signpost-D.
- **Each step:** the forward pass, the loss averaged over the three kinds (optionally weighted by `type_weights`), the
  backward pass, the clipper, then the optimizer step, unless the clipper skipped it.
- **Recorded:** the loss; the gradient norm before and after clipping; whether the step was skipped; how many traps were in
  the batch; and every `eval_every` steps, dev accuracy and KL.

In [ ]:
def train(loss_fn, make_clipper, steps=1500, lr=3e-3, batch=64, seed=0, eval_every=100, type_weights=None):
    torch.manual_seed(seed)
    model = Decider()
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    warm = int(0.15 * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, (steps - s) / (steps - warm)))
    clipper = make_clipper()
    weights = {k: (type_weights or {}).get(k, 1.0) for k in KINDS}
    rng = torch.Generator().manual_seed(seed)
    hist = {key: [] for key in ["loss", "pre", "post", "skipped", "traps", "dev_step", "dev_acc", "dev_kl"]}
    for step in range(1, steps + 1):
        idx = torch.randint(0, N_TRAIN, (batch,), generator=rng)
        out = model(train_set["x"][idx])
        loss = sum(weights[k] * loss_fn(out[k], train_set[k][idx], k) for k in KINDS) / sum(weights.values())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        pre, post, skipped = clipper(params)
        if not skipped:
            opt.step()
        sched.step()
        for key, value in [("loss", loss.item()), ("pre", pre), ("post", post), ("skipped", skipped),
                           ("traps", int(train_set["trap"][idx].sum()))]:
            hist[key].append(value)
        if step % eval_every == 0:
            r = evaluate(model, dev_set)
            hist["dev_step"].append(step)
            hist["dev_acc"].append(r["acc"])
            hist["dev_kl"].append(r["kl"])
    return model, hist

## 8. Experiments

### Experiment 1: the losses, with Signpost-D's clipper

Every loss trains a model from the same start, with the fixed norm limit of 1.0 that Signpost-D uses. Nine runs of 1,500
steps each; a few seconds each on a CPU. The table is the final dev scores, one row per loss.

In [ ]:
loss_runs = {name: train(fn, CLIPPERS["fixed_norm_1"]) for name, fn in LOSSES.items()}
loss_table = pd.DataFrame({name: evaluate(model, dev_set) for name, (model, _) in loss_runs.items()}).T
loss_table[["acc", "kl", "acc_clear", "kl_ambiguous", "score_acc", "score_level_err", "ece",
            "yesno_decisive", "yesno_decisive_right"]].round(3)

### Plot: how each loss learns

Dev accuracy (left) and dev KL (right) every 100 steps. Look for which losses learn fastest (cross-entropy family) and which
lag (JS, Brier: small gradients), and whether a loss that is better on accuracy pays for it in KL.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, h) in loss_runs.items():
    ax[0].plot(h["dev_step"], h["dev_acc"], label=name)
    ax[1].plot(h["dev_step"], h["dev_kl"], label=name)
ax[0].set(title="dev accuracy", xlabel="step")
ax[1].set(title="dev KL from gold (lower is better)", xlabel="step", yscale="log")
ax[0].legend(fontsize=8)
plt.tight_layout()

### Plot: the trade-offs

Three views of the final models:

1. **Right when clear, against honest when not:** accuracy on clear cases (higher is better) against KL on ambiguous cases
   (lower is better). The best losses sit at the top left.
2. **Decisive yes/no answers that are right**, the JevBench-style count.
3. **Score questions**: how far the average predicted level is from the gold's. Compare ce_expected_level with ce_emd.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for name, row in loss_table.iterrows():
    ax[0].scatter(row["kl_ambiguous"], row["acc_clear"])
    ax[0].annotate(name, (row["kl_ambiguous"], row["acc_clear"]), fontsize=8)
ax[0].set(xlabel="KL on ambiguous cases (lower is better)", ylabel="accuracy on clear cases")
loss_table["yesno_decisive_right"].plot.barh(ax=ax[1], title="yes/no: decisive and right")
loss_table["score_level_err"].plot.barh(ax=ax[2], title="score: |expected level error| (lower is better)")
plt.tight_layout()

### Experiment 2: the clippers, with soft cross-entropy

Now the loss is fixed (soft cross-entropy) and the clipper changes. The table adds two columns about the clipper itself: the
share of steps it clipped, and the share it skipped.

In [ ]:
clip_runs = {name: train(soft_ce, make) for name, make in CLIPPERS.items()}
rows = {}
for name, (model, h) in clip_runs.items():
    row = evaluate(model, dev_set)
    row["clipped_share"] = sum(post < pre - 1e-9 for pre, post in zip(h["pre"], h["post"])) / len(h["pre"])
    row["skipped_share"] = sum(h["skipped"]) / len(h["skipped"])
    rows[name] = row
clip_table = pd.DataFrame(rows).T
clip_table[["acc", "kl", "acc_clear", "kl_ambiguous", "ece", "clipped_share", "skipped_share"]].round(3)

### Plot: what each clipper does to the gradient norm

One panel per clipper: the norm before clipping (light) and after (dark), on a log scale, over the 1,500 steps. Look for:

- `none`: the spikes go straight through;
- `fixed_norm_1`: everything above 1 is flattened, which is nearly every step early on;
- `autoclip_p10`: a limit that drifts down with the run;
- `zclip_2.5`: only the spikes are cut, the normal steps keep their size;
- `skip_spikes_4sd`: skipped steps drop to zero after clipping;
- `value_0.05`, `agc_0.05`: different shapes, because they clip per element or per tensor.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5), sharex=True)
for ax, (name, (_, h)) in zip(axes.flat, clip_runs.items()):
    ax.plot(h["pre"], lw=0.6, alpha=0.4, label="before")
    ax.plot([max(v, 1e-4) for v in h["post"]], lw=0.6, label="after")
    ax.set(title=name, yscale="log")
axes.flat[-1].axis("off")
axes.flat[0].legend(fontsize=8)
fig.supxlabel("step")
fig.supylabel("gradient norm")
plt.tight_layout()

### Plot: where the spikes come from

With no clipping, each step's gradient norm against the number of trap cases in its batch. If the traps cause the spikes,
the norm should rise with the trap count. This is why a statistical clipper (ZClip, skipping) is attractive: a spike is
usually a batch the model should learn little from.

In [ ]:
h = clip_runs["none"][1]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(h["traps"], h["pre"], s=6, alpha=0.4)
ax.set(xlabel="trap cases in the batch", ylabel="gradient norm (no clipping)", yscale="log",
       title="gradient spikes against traps in the batch")
plt.tight_layout()

### Experiment 3: losses and clippers together

The two choices interact, so here is a small grid: four losses (Signpost-D's, its base, and the two candidates on our to-do
list) times four clippers. Sixteen runs.

In [ ]:
GRID_LOSSES = ["soft_ce", "ce_expected_level", "ce_emd", "ce_hard"]
GRID_CLIPPERS = ["none", "fixed_norm_1", "autoclip_p10", "zclip_2.5"]
grid = {(l, c): evaluate(train(LOSSES[l], CLIPPERS[c])[0], dev_set) for l in GRID_LOSSES for c in GRID_CLIPPERS}

### Plot: the grid as heatmaps

Rows are losses, columns are clippers; each cell is printed with its value. Left: dev accuracy (higher is better). Right:
dev KL (lower is better). A loss whose row is good under every clipper is robust; a clipper whose column is good for every
loss is a safe default.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, metric, cmap in [(ax[0], "acc", "viridis"), (ax[1], "kl", "viridis_r")]:
    m = torch.tensor([[grid[(l, c)][metric] for c in GRID_CLIPPERS] for l in GRID_LOSSES])
    a.imshow(m, cmap=cmap)
    a.set(xticks=range(len(GRID_CLIPPERS)), xticklabels=GRID_CLIPPERS, yticks=range(len(GRID_LOSSES)),
          yticklabels=GRID_LOSSES, title=f"dev {metric}")
    for i in range(len(GRID_LOSSES)):
        for j in range(len(GRID_CLIPPERS)):
            a.text(j, i, f"{m[i, j]:.3f}", ha="center", va="center", color="white", fontsize=9)
plt.tight_layout()

### Experiment 4: clipping and the learning rate

When nearly every step is clipped, the clip limit and the learning rate together set the step length, so changing one is
like changing the other. Here soft cross-entropy trains at three learning rates with the fixed limit and with ZClip.

In [ ]:
LRS = [1e-3, 3e-3, 1e-2]
lr_rows = {(c, lr): evaluate(train(soft_ce, CLIPPERS[c], lr=lr)[0], dev_set)
           for c in ["fixed_norm_1", "zclip_2.5"] for lr in LRS}

### Plot: accuracy against learning rate, per clipper

If the fixed limit is acting as the step-size control, its curve moves differently with the learning rate than ZClip's,
whose normal steps keep their natural size: ZClip usually wants a lower learning rate for the same effect.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c in ["fixed_norm_1", "zclip_2.5"]:
    ax[0].plot(LRS, [lr_rows[(c, lr)]["acc"] for lr in LRS], marker="o", label=c)
    ax[1].plot(LRS, [lr_rows[(c, lr)]["kl"] for lr in LRS], marker="o", label=c)
ax[0].set(xscale="log", xlabel="peak learning rate", title="dev accuracy")
ax[1].set(xscale="log", xlabel="peak learning rate", title="dev KL (lower is better)")
ax[0].legend()
plt.tight_layout()

## 9. What to take away

- **Soft cross-entropy is the right base.** It copies the gold, decisive or hedged, and its gradient stays strong on confident
  mistakes. Brier and JS are gentler and slower.
- **Losses that are more decisive than the gold** (`ce_hard`, `ce_margin`) trade a little KL on hedged cases for accuracy and
  decisive yes/no answers, the direction JevBench rewards. Keep λ small.
- **For ordered scales, the earth mover's distance uses the order properly.** The expected-level term can be satisfied by a
  prediction split between the two ends.
- **A fixed clip below the typical gradient norm clips every step.** It then acts as the step size, interacting with the
  learning rate. Statistical clippers (ZClip, AutoClip, skipping) touch only the outliers, which in this data are the traps.
- **Results on a toy model are intuition, not proof.** Any change carried over to Signpost-D is tested the usual way: one
  change at a time, one seed, against a measured baseline.